## Imports

In [8]:
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [1]:
df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_2021_2026.csv")
print(f"Loaded {len(df)} rows")
df.head()

Loaded 282425 rows


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,e7e2f23e-2b81-4105-9e3b-675b795d1626,M,https://play-lh.googleusercontent.com/a/ACg8oc...,sign up bonus completely failed. email not ver...,1,0,NaN,2026-07-23 14:33:57,Hi Thank you for your feedback We apologize fo...,2026-07-23 14:39:03,NaN
1,b964471f-a769-4518-a3fe-9d43139ba4a8,J T (Savvy Eclectic),https://play-lh.googleusercontent.com/a-/ALV-U...,"Started off ok, but I've been hearing VERY CON...",3,0,10.139,2026-07-23 14:33:30,Hi. We are sorry to hear about your concerns. ...,2026-07-23 14:38:56,10.139
2,4b090baa-df35-4463-869a-34c1f7c65f44,Murphy,https://play-lh.googleusercontent.com/a-/ALV-U...,good,5,0,10.126.1,2026-07-23 14:27:28,NaN,NaN,10.126.1
3,1b0847ed-dfd4-4f9e-ac1a-8035b139e8e7,Christopher Mugara,https://play-lh.googleusercontent.com/a/ACg8oc...,Great,5,0,10.139,2026-07-23 14:25:39,NaN,NaN,10.139
4,b587b252-cb43-456e-b74c-cff89f74bf21,Ronie Boy Consolacion,https://play-lh.googleusercontent.com/a-/ALV-U...,safeguard your money,5,0,10.129.1,2026-07-23 13:31:44,NaN,NaN,10.129.1


In [2]:
df.isnull().sum()

reviewId                     0
userName                     3
userImage                    0
content                     22
score                        0
thumbsUpCount                0
reviewCreatedVersion     32455
at                           0
replyContent            257725
repliedAt               257725
appVersion               32455
dtype: int64

## Handle missing values

In [3]:
df["userName"] = df["userName"].fillna("Anonymous")
df["reviewCreatedVersion"] = df["reviewCreatedVersion"].fillna("Unknown")
df["appVersion"] = df["appVersion"].fillna("Unknown")
df["got_reply"] = df["replyContent"].notnull()

before = len(df)
df = df.dropna(subset=["content"])
df = df[df["content"].str.strip() != ""]
after = len(df)
print(f"Dropped {before - after} rows with empty review text")
print(f"Remaining rows: {after}")

Dropped 22 rows with empty review text
Remaining rows: 282403


## Drop Duplicate Columns

In [4]:
before = len(df)
df = df.drop_duplicates(subset=["reviewId"])
after = len(df)
print(f"Dropped {before - after} duplicate rows")

Dropped 0 duplicate rows


## Rename columns and parse dates

In [5]:
df = df.rename(columns={
    "content": "review_text",
    "score": "rating",
    "at": "review_date",
    "thumbsUpCount": "thumbs_up",
})

df["review_date"] = pd.to_datetime(df["review_date"])
df["year"] = df["review_date"].dt.year
df["month"] = df["review_date"].dt.month
df["review_length"] = df["review_text"].str.len()

df[["review_date", "year", "rating"]].head()

,review_date,year,rating
0,2026-07-23 14:33:57,2026,1
1,2026-07-23 14:33:30,2026,3
2,2026-07-23 14:27:28,2026,5
3,2026-07-23 14:25:39,2026,5
4,2026-07-23 13:31:44,2026,5


## Sanity check

In [6]:
print(f"Final row count: {len(df)}")
print(f"\nRating distribution:")
print(df["rating"].value_counts().sort_index())
print(f"\nReviews per year:")
print(df["year"].value_counts().sort_index())

Final row count: 282403

Rating distribution:
rating
1     34140
2      4808
3      5641
4     20425
5    217389
Name: count, dtype: int64

Reviews per year:
year
2015      153
2016     1723
2017     4501
2018    16737
2019    41197
2020    32985
2021    33489
2022    31591
2023    33327
2024    37820
2025    31606
2026    17274
Name: count, dtype: int64


## Save

In [7]:
df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended_clean.csv", index=False)
print("Saved.")

Saved.


## Working with cleaned data

In [9]:
analyzer = SentimentIntensityAnalyzer()
df = pd.read_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended_clean.csv")
print(f"Loaded {len(df)} rows")

Loaded 282403 rows


## Sentiment scoring

In [10]:
def get_sentiment_score(text):
    return analyzer.polarity_scores(str(text))["compound"]

df["sentiment_score"] = df["review_text"].apply(get_sentiment_score)

def score_to_label(score):
    if score >= 0.05:
        return "positive"
    elif score <= -0.05:
        return "negative"
    else:
        return "neutral"

df["sentiment_label"] = df["sentiment_score"].apply(score_to_label)
df["sentiment_label"].value_counts()

sentiment_label
positive    221073
neutral      32380
negative     28950
Name: count, dtype: int64

## Theme tagging (same refined themes as before)

In [12]:
themes = {
    "account_freeze": ["freeze", "frozen", "block", "blocked", "locked", "lock"],
    "customer_support": ["support", "customer service", "help desk", "chatbot", "agent", "response time"],
    "fees": ["fee", "charge", "charged", "expensive", "cost", "commission"],
    "app_stability": ["bug", "crash", "glitch", "freezing app", "not working", "won't open", "keeps closing"],
    "app_ux_positive": ["easy to use", "user friendly", "interface", "design", "simple", "intuitive"],
    "transfers": ["transfer", "payment", "send money", "withdraw", "deposit"],
    "verification": ["verify", "verification", "kyc", "id check", "document"],
}

def tag_themes(text):
    text = str(text).lower()
    matched = [theme for theme, keywords in themes.items() if any(kw in text for kw in keywords)]
    return matched if matched else ["other"]

df["themes"] = df["review_text"].apply(tag_themes)

## Save

In [13]:
df.to_csv("E:/project/Revolut-analysis-deashboard/Data/revolut_reviews_extended_analyzed.csv", index=False)
print(f"Saved {len(df)} analyzed reviews.")

Saved 282403 analyzed reviews.
